In [1]:
from pathlib import Path

import pandas as pd


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder: {DATA_PROCESSED}")

Project root: /Users/lohith/Documents/GitHub/UK-Yield-Curve-Recession-Forecasting
Processed data folder: /Users/lohith/Documents/GitHub/UK-Yield-Curve-Recession-Forecasting/data/processed


In [2]:
yields = pd.read_csv(
    DATA_PROCESSED / "boe_quarterly_yields.csv"
)

gdp = pd.read_csv(
    DATA_PROCESSED / "ons_quarterly_real_gdp.csv"
)

yields["quarter"] = pd.PeriodIndex(yields["quarter"], freq="Q")
gdp["quarter"] = pd.PeriodIndex(gdp["quarter"], freq="Q")

print(f"Yield observations: {len(yields)}")
print(f"GDP observations: {len(gdp)}")

display(yields.head())
display(gdp.head())

Yield observations: 191
GDP observations: 286


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days
0,1979Q1,11.827995,12.679240,0.851246,64
1,1979Q2,10.803278,11.498352,0.695075,61
2,1979Q3,11.717763,11.913749,0.195985,64
3,1979Q4,13.503208,13.197965,-0.305243,64
4,1980Q1,14.502812,13.655117,-0.847694,64


,quarter,gdp_real_mn
0,1955Q1,145457
1,1955Q2,145551
2,1955Q3,147995
3,1955Q4,147119
4,1956Q1,148955


In [3]:
print(f"Duplicate yield quarters: {yields['quarter'].duplicated().sum()}")
print(f"Duplicate GDP quarters: {gdp['quarter'].duplicated().sum()}")

print(f"Yield period: {yields['quarter'].min()} to {yields['quarter'].max()}")
print(f"GDP period: {gdp['quarter'].min()} to {gdp['quarter'].max()}")

Duplicate yield quarters: 0
Duplicate GDP quarters: 0
Yield period: 1979Q1 to 2026Q3
GDP period: 1955Q1 to 2026Q2


In [4]:
merge_audit = (
    yields.merge(
        gdp,
        on="quarter",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values("quarter")
    .reset_index(drop=True)
)

print(merge_audit["_merge"].value_counts())

display(
    merge_audit.loc[
        merge_audit["_merge"] != "both",
        ["quarter", "_merge"],
    ].tail(10)
)

_merge
both          190
right_only     96
left_only       1
Name: count, dtype: int64


,quarter,_merge
87,1976Q4,right_only
88,1977Q1,right_only
89,1977Q2,right_only
90,1977Q3,right_only
91,1977Q4,right_only
92,1978Q1,right_only
93,1978Q2,right_only
94,1978Q3,right_only
95,1978Q4,right_only
286,2026Q3,left_only


In [5]:
merged = (
    yields.merge(
        gdp,
        on="quarter",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("quarter")
    .reset_index(drop=True)
)

print(f"Number of merged quarters: {len(merged)}")
print(f"First merged quarter: {merged['quarter'].min()}")
print(f"Last merged quarter: {merged['quarter'].max()}")

print("\nMissing values:")
print(merged.isna().sum())

display(merged.head())
display(merged.tail())

Number of merged quarters: 190
First merged quarter: 1979Q1
Last merged quarter: 2026Q2

Missing values:
quarter          0
yield_2y         0
yield_10y        0
spread_10y_2y    0
trading_days     0
gdp_real_mn      0
dtype: int64


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days,gdp_real_mn
0,1979Q1,11.827995,12.679240,0.851246,64,284457
1,1979Q2,10.803278,11.498352,0.695075,61,296841
2,1979Q3,11.717763,11.913749,0.195985,64,290297
3,1979Q4,13.503208,13.197965,-0.305243,64,293265
4,1980Q1,14.502812,13.655117,-0.847694,64,290378


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days,gdp_real_mn
185,2025Q2,3.753570,4.589065,0.835495,61,704274
186,2025Q3,3.755126,4.679847,0.924721,65,704802
187,2025Q4,3.680211,4.584931,0.904720,64,705160
188,2026Q1,3.725403,4.623884,0.898481,63,709598
189,2026Q2,4.193059,4.935797,0.742738,61,712545


In [6]:
gdp_with_growth = (
    gdp.sort_values("quarter")
    .reset_index(drop=True)
    .copy()
)

gdp_with_growth["gdp_growth_qoq_pct"] = (
    gdp_with_growth["gdp_real_mn"]
    .pct_change(fill_method=None)
    .mul(100)
)

display(gdp_with_growth.head())

,quarter,gdp_real_mn,gdp_growth_qoq_pct
0,1955Q1,145457,NaN
1,1955Q2,145551,0.064624
2,1955Q3,147995,1.679137
3,1955Q4,147119,-0.591912
4,1956Q1,148955,1.247969


In [7]:
negative_growth = gdp_with_growth["gdp_growth_qoq_pct"] < 0

adjacent_negative_growth = (
    negative_growth.shift(1, fill_value=False)
    | negative_growth.shift(-1, fill_value=False)
)

gdp_with_growth["recession"] = (
    negative_growth & adjacent_negative_growth
).astype(int)

display(
    gdp_with_growth[
        ["quarter", "gdp_real_mn", "gdp_growth_qoq_pct", "recession"]
    ].head(10)
)

,quarter,gdp_real_mn,gdp_growth_qoq_pct,recession
0,1955Q1,145457,NaN,0
1,1955Q2,145551,0.064624,0
2,1955Q3,147995,1.679137,0
3,1955Q4,147119,-0.591912,0
4,1956Q1,148955,1.247969,0
5,1956Q2,148835,-0.080561,1
6,1956Q3,148665,-0.114220,1
7,1956Q4,149607,0.633639,0
8,1957Q1,152331,1.820770,0
9,1957Q2,152285,-0.030197,1


In [8]:
contraction_check = gdp_with_growth.loc[
    negative_growth,
    ["quarter", "gdp_growth_qoq_pct", "recession"],
].copy()

print(f"Negative-growth quarters: {len(contraction_check)}")
print(
    "Quarters classified as recession:",
    gdp_with_growth["recession"].sum(),
)

display(contraction_check)

Negative-growth quarters: 54
Quarters classified as recession: 32


,quarter,gdp_growth_qoq_pct,recession
3,1955Q4,-0.591912,0
5,1956Q2,-0.080561,1
6,1956Q3,-0.114220,1
9,1957Q2,-0.030197,1
10,1957Q3,-0.620547,1
13,1958Q2,-2.488444,0
21,1960Q2,-0.689509,0
26,1961Q3,-0.481889,1
27,1961Q4,-0.204424,1
31,1962Q4,-0.355089,0


In [10]:
recession_start = (
    gdp_with_growth["recession"].eq(1)
    & gdp_with_growth["recession"].shift(1, fill_value=0).eq(0)
)

print(f"Number of recession episodes: {recession_start.sum()}")

display(
    gdp_with_growth.loc[
        recession_start,
        ["quarter", "gdp_growth_qoq_pct"],
    ]
)

Number of recession episodes: 11


,quarter,gdp_growth_qoq_pct
5,1956Q2,-0.080561
9,1957Q2,-0.030197
26,1961Q3,-0.481889
74,1973Q3,-0.933011
81,1975Q2,-1.619814
100,1980Q1,-0.984434
142,1990Q3,-1.090584
148,1992Q1,-0.010714
213,2008Q2,-0.536740
260,2020Q1,-2.744909


In [11]:
model_data = (
    yields.merge(
        gdp_with_growth[
            [
                "quarter",
                "gdp_real_mn",
                "gdp_growth_qoq_pct",
                "recession",
            ]
        ],
        on="quarter",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("quarter")
    .reset_index(drop=True)
)

print(f"Number of observations: {len(model_data)}")
print(f"First quarter: {model_data['quarter'].min()}")
print(f"Last quarter: {model_data['quarter'].max()}")

print("\nMissing values:")
print(model_data.isna().sum())

display(model_data.head())
display(model_data.tail())

Number of observations: 190
First quarter: 1979Q1
Last quarter: 2026Q2

Missing values:
quarter               0
yield_2y              0
yield_10y             0
spread_10y_2y         0
trading_days          0
gdp_real_mn           0
gdp_growth_qoq_pct    0
recession             0
dtype: int64


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days,gdp_real_mn,gdp_growth_qoq_pct,recession
0,1979Q1,11.827995,12.679240,0.851246,64,284457,-0.300371,0
1,1979Q2,10.803278,11.498352,0.695075,61,296841,4.353558,0
2,1979Q3,11.717763,11.913749,0.195985,64,290297,-2.204547,0
3,1979Q4,13.503208,13.197965,-0.305243,64,293265,1.022401,0
4,1980Q1,14.502812,13.655117,-0.847694,64,290378,-0.984434,1


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days,gdp_real_mn,gdp_growth_qoq_pct,recession
185,2025Q2,3.753570,4.589065,0.835495,61,704274,0.155864,0
186,2025Q3,3.755126,4.679847,0.924721,65,704802,0.074971,0
187,2025Q4,3.680211,4.584931,0.904720,64,705160,0.050794,0
188,2026Q1,3.725403,4.623884,0.898481,63,709598,0.629361,0
189,2026Q2,4.193059,4.935797,0.742738,61,712545,0.415306,0


In [12]:
assert len(model_data) == 190
assert model_data["quarter"].is_unique
assert model_data["quarter"].is_monotonic_increasing
assert model_data["gdp_growth_qoq_pct"].notna().all()
assert model_data["recession"].isin([0, 1]).all()

print("All Phase 3B checks passed.")

All Phase 3B checks passed.


In [13]:
model_recession_start = (
    model_data["recession"].eq(1)
    & model_data["recession"].shift(1, fill_value=0).eq(0)
)

print(
    "Recession quarters in modelling sample:",
    model_data["recession"].sum(),
)

print(
    "Recession episodes in modelling sample:",
    model_recession_start.sum(),
)

display(
    model_data.loc[
        model_recession_start,
        ["quarter", "gdp_growth_qoq_pct"],
    ]
)

Recession quarters in modelling sample: 21
Recession episodes in modelling sample: 6


,quarter,gdp_growth_qoq_pct
4,1980Q1,-0.984434
46,1990Q3,-1.090584
52,1992Q1,-0.010714
117,2008Q2,-0.536740
164,2020Q1,-2.744909
178,2023Q3,-0.239597


In [14]:
HORIZONS = [1, 2, 4, 6]

for horizon in HORIZONS:
    column_name = f"recession_h{horizon}"

    model_data[column_name] = (
        model_data["recession"]
        .shift(-horizon)
        .astype("Int64")
    )

In [15]:
example = model_data.loc[
    model_data["quarter"] == pd.Period("1979Q4", freq="Q"),
    [
        "quarter",
        "spread_10y_2y",
        "recession",
        "recession_h1",
        "recession_h2",
        "recession_h4",
        "recession_h6",
    ],
]

display(example)

,quarter,spread_10y_2y,recession,recession_h1,recession_h2,recession_h4,recession_h6
3,1979Q4,-0.305243,0,1,1,1,0


In [16]:
target_summary = pd.DataFrame(
    {
        "horizon": HORIZONS,
        "valid_observations": [
            model_data[f"recession_h{h}"].notna().sum()
            for h in HORIZONS
        ],
        "missing_future_outcomes": [
            model_data[f"recession_h{h}"].isna().sum()
            for h in HORIZONS
        ],
        "recession_outcomes": [
            model_data[f"recession_h{h}"].sum()
            for h in HORIZONS
        ],
    }
)

display(target_summary)

,horizon,valid_observations,missing_future_outcomes,recession_outcomes
0,1,189,1,21
1,2,188,2,21
2,4,186,4,21
3,6,184,6,19


In [17]:
for horizon in HORIZONS:
    target = model_data[f"recession_h{horizon}"]

    assert target.isna().sum() == horizon
    assert target.dropna().isin([0, 1]).all()

print("All Phase 3C checks passed.")

All Phase 3C checks passed.


In [18]:
FINAL_COLUMNS = [
    "quarter",
    "yield_2y",
    "yield_10y",
    "spread_10y_2y",
    "trading_days",
    "gdp_real_mn",
    "gdp_growth_qoq_pct",
    "recession",
    "recession_h1",
    "recession_h2",
    "recession_h4",
    "recession_h6",
]

final_data = model_data[FINAL_COLUMNS].copy()

display(final_data.head())
display(final_data.tail())

,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days,gdp_real_mn,gdp_growth_qoq_pct,recession,recession_h1,recession_h2,recession_h4,recession_h6
0,1979Q1,11.827995,12.679240,0.851246,64,284457,-0.300371,0,0,0,1,1
1,1979Q2,10.803278,11.498352,0.695075,61,296841,4.353558,0,0,0,1,1
2,1979Q3,11.717763,11.913749,0.195985,64,290297,-2.204547,0,0,1,1,1
3,1979Q4,13.503208,13.197965,-0.305243,64,293265,1.022401,0,1,1,1,0
4,1980Q1,14.502812,13.655117,-0.847694,64,290378,-0.984434,1,1,1,1,0


,quarter,yield_2y,yield_10y,spread_10y_2y,trading_days,gdp_real_mn,gdp_growth_qoq_pct,recession,recession_h1,recession_h2,recession_h4,recession_h6
185,2025Q2,3.753570,4.589065,0.835495,61,704274,0.155864,0,0,0,0,<NA>
186,2025Q3,3.755126,4.679847,0.924721,65,704802,0.074971,0,0,0,<NA>,<NA>
187,2025Q4,3.680211,4.584931,0.904720,64,705160,0.050794,0,0,0,<NA>,<NA>
188,2026Q1,3.725403,4.623884,0.898481,63,709598,0.629361,0,0,<NA>,<NA>,<NA>
189,2026Q2,4.193059,4.935797,0.742738,61,712545,0.415306,0,<NA>,<NA>,<NA>,<NA>


In [19]:
expected_quarters = pd.period_range(
    start=final_data["quarter"].min(),
    end=final_data["quarter"].max(),
    freq="Q",
)

assert len(final_data) == 190
assert final_data["quarter"].is_unique
assert final_data["quarter"].is_monotonic_increasing
assert final_data["quarter"].tolist() == expected_quarters.tolist()

print("Quarterly structure is correct.")

Quarterly structure is correct.


In [20]:
recalculated_spread = (
    final_data["yield_10y"] - final_data["yield_2y"]
)

maximum_spread_error = (
    final_data["spread_10y_2y"] - recalculated_spread
).abs().max()

print(f"Maximum spread calculation error: {maximum_spread_error}")

assert maximum_spread_error < 1e-10

print("Yield spreads are correct.")

Maximum spread calculation error: 3.552713678800501e-15
Yield spreads are correct.


In [21]:
recalculated_growth = (
    final_data["gdp_real_mn"]
    .pct_change(fill_method=None)
    .mul(100)
)

maximum_growth_error = (
    final_data["gdp_growth_qoq_pct"].iloc[1:]
    - recalculated_growth.iloc[1:]
).abs().max()

print(f"Maximum GDP-growth calculation error: {maximum_growth_error}")

assert maximum_growth_error < 1e-10

print("GDP growth rates are correct.")

Maximum GDP-growth calculation error: 0.0
GDP growth rates are correct.


In [22]:
full_negative_growth = (
    gdp_with_growth["gdp_growth_qoq_pct"] < 0
)

expected_recession = (
    full_negative_growth
    & (
        full_negative_growth.shift(1, fill_value=False)
        | full_negative_growth.shift(-1, fill_value=False)
    )
).astype(int)

assert gdp_with_growth["recession"].equals(expected_recession)

print("Recession indicator is correct.")

Recession indicator is correct.


In [23]:
for horizon in HORIZONS:
    target_column = f"recession_h{horizon}"

    expected_target = (
        final_data["recession"]
        .shift(-horizon)
        .astype("Int64")
    )

    assert final_data[target_column].equals(expected_target)
    assert final_data[target_column].isna().sum() == horizon
    assert final_data[target_column].dropna().isin([0, 1]).all()

print("All future recession targets are correctly aligned.")

All future recession targets are correctly aligned.


In [24]:
print(f"Rows: {len(final_data)}")
print(f"Columns: {len(final_data.columns)}")
print(f"Period: {final_data['quarter'].min()} to {final_data['quarter'].max()}")
print(f"Current recession quarters: {final_data['recession'].sum()}")

print("\nMissing values:")
print(final_data.isna().sum())

print("\nColumn types:")
print(final_data.dtypes)

Rows: 190
Columns: 12
Period: 1979Q1 to 2026Q2
Current recession quarters: 21

Missing values:
quarter               0
yield_2y              0
yield_10y             0
spread_10y_2y         0
trading_days          0
gdp_real_mn           0
gdp_growth_qoq_pct    0
recession             0
recession_h1          1
recession_h2          2
recession_h4          4
recession_h6          6
dtype: int64

Column types:
quarter               period[Q-DEC]
yield_2y                    float64
yield_10y                   float64
spread_10y_2y               float64
trading_days                  int64
gdp_real_mn                   int64
gdp_growth_qoq_pct          float64
recession                     int64
recession_h1                  Int64
recession_h2                  Int64
recession_h4                  Int64
recession_h6                  Int64
dtype: object


In [25]:
OUTPUT_PATH = (
    DATA_PROCESSED
    / "uk_yield_curve_recession_dataset.csv"
)

final_data.to_csv(OUTPUT_PATH, index=False)

print(f"Saved dataset to: {OUTPUT_PATH}")

Saved dataset to: /Users/lohith/Documents/GitHub/UK-Yield-Curve-Recession-Forecasting/data/processed/uk_yield_curve_recession_dataset.csv


In [26]:
saved_data = pd.read_csv(OUTPUT_PATH)

assert saved_data.shape == final_data.shape
assert saved_data.columns.tolist() == FINAL_COLUMNS

print(f"Saved rows: {saved_data.shape[0]}")
print(f"Saved columns: {saved_data.shape[1]}")
print("Saved CSV verified successfully.")

Saved rows: 190
Saved columns: 12
Saved CSV verified successfully.
